# 📈 S&P 500 Direction Prediction

**Goal:** Predict whether the S&P 500 will rise or fall the next day using macroeconomic indicators and market data.

**Approach:** Combine FRED (Federal Reserve Economic Data) macro indicators with yfinance market data, engineer features, and train a Random Forest classifier with SMOTE oversampling and hyperparameter tuning.

---

### 📁 Data Sources
| Source | Data | Period |
|---|---|---|
| [yfinance](https://pypi.org/project/yfinance/) | S&P500, US10Y, WTI Oil, Gold, Copper, VIX, etc. | 2015–2024 |
| [FRED API](https://fred.stlouisfed.org/) | CPI, Fed Funds Rate, Unemployment, Retail Sales, PCE, Inflation Expectation | 2015–2024 |

### 📌 Table of Contents
1. [Setup & Data Collection](#1-setup--data-collection)
2. [EDA — Correlation Analysis](#2-eda--correlation-analysis)
3. [Feature Engineering](#3-feature-engineering)
4. [Model Training — Random Forest](#4-model-training--random-forest)
5. [Model Training — Logistic Regression](#5-model-training--logistic-regression)
6. [Results](#6-results)


## 1. Setup & Data Collection

Install required libraries and load market + macro data.

> ⚠️ **FRED API Key required.** Get a free key at [fred.stlouisfed.org/docs/api/api_key.html](https://fred.stlouisfed.org/docs/api/api_key.html) and set it as an environment variable:
> ```bash
> export FRED_API_KEY="your_key_here"
> ```


In [ ]:
!pip install yfinance fredapi imbalanced-learn

In [ ]:
# 1. Import libraries
import os
import time
import yfinance as yf
from fredapi import Fred
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             classification_report, confusion_matrix,
                             roc_curve, auc)
from imblearn.over_sampling import SMOTE

# 2. FRED API key — set via environment variable (never hardcode)
FRED_API_KEY = os.environ.get("FRED_API_KEY", "your_fred_api_key_here")
fred = Fred(api_key=FRED_API_KEY)

# 3. Market data tickers
market_tickers = {
    "S&P500":  "^GSPC",
    "US10Y":   "^TNX",
    "WTI_Oil": "CL=F",
    "Copper":  "HG=F",
    "Gold":    "GC=F",
    "NatGas":  "NG=F",
    "Corn":    "ZC=F",
    "VIX":     "^VIX"
}

# 4. Download market data (2015–2024)
market_df = yf.download(
    list(market_tickers.values()),
    start="2015-01-01",
    end="2024-12-31"
)["Close"]
market_df.columns = list(market_tickers.keys())
market_df = market_df.dropna()
market_df.index.name = "Date"

# 5. FRED macro indicators
macro_indicators = {
    "CPI":                    "CPIAUCSL",
    "FedFundsRate":           "FEDFUNDS",
    "UnemploymentRate":       "UNRATE",
    "RetailSales":            "RSAFS",
    "InflationExpectation10Y":"T10YIE",
    "PCE":                    "PCE"
}

# 6. Collect FRED data and forward-fill to daily frequency
macro_data_list = []
for name, code_id in macro_indicators.items():
    try:
        series = fred.get_series(code_id, start="2015-01-01", end="2024-12-31")
        if series is not None and not series.empty:
            df_s = series.to_frame(name)
            df_s.index = pd.to_datetime(df_s.index)
            df_s = df_s.resample("D").ffill()
            macro_data_list.append(df_s)
    except Exception as e:
        print(f"❌ {name} error: {e}")

macro_df = pd.concat(macro_data_list, axis=1)
macro_df.index.name = "Date"
macro_df = macro_df.reindex(market_df.index, method="ffill")

# 7. Merge into single dataframe
df = pd.concat([market_df, macro_df], axis=1)

print("📌 Dataset shape:", df.shape)
print("\n📌 Missing values:\n", df.isna().sum())
df.head()

## 2. EDA — Correlation Analysis

Explore relationships between S&P 500 and macro/market variables.

In [ ]:
# Correlation heatmap
corr_matrix = df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    annot=True, fmt=".2f",
    cmap="coolwarm", center=0,
    square=True, linewidths=0.5,
    cbar_kws={"shrink": 0.75}
)
plt.title("Correlation Matrix", fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# S&P500 correlation ranking
print("\n📊 Correlation with S&P500:")
print(df.corr()["S&P500"].sort_values(ascending=False))

### Key Variable Pair Charts
Dual-axis line charts to visualise co-movement between S&P 500 and each indicator.

In [ ]:
pairs = {
    "US10Y":                  "US 10Y Yield (%)",
    "WTI_Oil":                "WTI Oil ($/bbl)",
    "Gold":                   "Gold ($/oz)",
    "InflationExpectation10Y":"10Y Inflation Expectation (%)",
    "CPI":                    "CPI",
    "UnemploymentRate":       "Unemployment Rate (%)",
    "FedFundsRate":           "Fed Funds Rate (%)"
}

fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()

for idx, (col, label) in enumerate(pairs.items()):
    ax1 = axes[idx]
    ax1.set_xlabel("Date")
    ax1.set_ylabel("S&P500", color="tab:blue")
    ax1.plot(df.index, df["S&P500"], color="tab:blue", linewidth=1)
    ax1.tick_params(axis='y', labelcolor="tab:blue")

    ax2 = ax1.twinx()
    ax2.set_ylabel(label, color="tab:red")
    ax2.plot(df.index, df[col], color="tab:red", linewidth=1)
    ax2.tick_params(axis='y', labelcolor="tab:red")

    ax1.set_title(f"S&P500 vs {col}", fontsize=11)

# Hide unused subplot
if len(pairs) < len(axes):
    axes[-1].set_visible(False)

plt.tight_layout()
plt.savefig("sp500_dual_axis_charts.png", bbox_inches="tight")
plt.show()

## 3. Feature Engineering

Create model-ready features:
- **Target:** Next-day return > 0.2% → 1 (Up), else 0 (Down)
- **Macro change rates:** pct_change for CPI, Retail, WTI, PCE, Gold, Unemployment
- **Technical indicators:** 10-day MA, 5-day volatility, 5-day momentum
- **Lag feature:** Retail Sales change (lag 1)


In [ ]:
# Target: next-day return > 0.2%
df["Return"] = df["S&P500"].pct_change().shift(-1)
df["Target"] = (df["Return"] > 0.002).astype(int)
df = df[:-1]  # drop last row (no return available)

# Macro change rates
df["Inflation_change"]   = df["InflationExpectation10Y"].pct_change()
df["Retail_change"]      = df["RetailSales"].pct_change()
df["WTI_change"]         = df["WTI_Oil"].pct_change()
df["PCE_change"]         = df["PCE"].pct_change()
df["Gold_change"]        = df["Gold"].pct_change()
df["CPI_change"]         = df["CPI"].pct_change()
df["Unemp_change"]       = df["UnemploymentRate"].pct_change()

# Technical indicators
df["MA10"]               = df["S&P500"].rolling(window=10).mean()
df["Above_MA10"]         = (df["S&P500"] > df["MA10"]).astype(int)
df["Return_3d_avg"]      = df["Return"].rolling(window=3).mean()
df["Volatility_5d"]      = df["Return"].rolling(5).std()
df["Momentum_5d"]        = df["Return"].rolling(5).sum()
df["Retail_change_lag1"] = df["Retail_change"].shift(1)

df = df.dropna().reset_index(drop=True)

# Define feature columns
feature_cols = [
    "Inflation_change", "Retail_change", "WTI_change",
    "PCE_change", "Gold_change", "CPI_change", "Unemp_change",
    "Above_MA10", "Return_3d_avg", "Volatility_5d",
    "Momentum_5d", "Retail_change_lag1"
]

X_raw = df[feature_cols]
y     = df["Target"]

print(f"Dataset: {X_raw.shape}, Target balance:\n{y.value_counts()}")

### Train/Test Split & Scaling

In [ ]:
# Time-series split (no shuffle)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, shuffle=False)

# StandardScaler
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_raw),
    columns=X_train_raw.columns, index=X_train_raw.index)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_raw),
    columns=X_test_raw.columns, index=X_test_raw.index)

# SMOTE oversampling to handle class imbalance
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f"Train (after SMOTE): {X_train_resampled.shape}")
print(f"Test:                {X_test_scaled.shape}")
print(f"Target balance after SMOTE:\n{pd.Series(y_train_resampled).value_counts()}")

## 4. Model Training — Random Forest

Use `RandomizedSearchCV` to find optimal hyperparameters, then evaluate on the held-out test set.

In [ ]:
start_time = time.time()

# Hyperparameter search space
param_dist = {
    "n_estimators":     [2000],
    "max_depth":        [20],
    "min_samples_split":[2],
    "min_samples_leaf": [1],
    "max_features":     [None],
    "class_weight":     ["balanced"]
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, class_weight="balanced"),
    param_distributions=param_dist,
    n_iter=30,
    scoring="f1",
    cv=3,
    n_jobs=1,
    random_state=42
)
random_search.fit(X_train_resampled, y_train_resampled)

# Best model
best_rf = random_search.best_estimator_
y_pred_rf = best_rf.predict(X_test_scaled)

# Elapsed time
elapsed = time.time() - start_time
elapsed_str = f"{int(elapsed//60)}m {int(elapsed%60)}s"

# Metrics
rf_results = pd.DataFrame([{
    "Model":        "Random Forest",
    "Accuracy":     round(accuracy_score(y_test, y_pred_rf), 4),
    "Precision":    round(precision_score(y_test, y_pred_rf, zero_division=0), 4),
    "Recall":       round(recall_score(y_test, y_pred_rf, zero_division=0), 4),
    "F1 Score":     round(f1_score(y_test, y_pred_rf, zero_division=0), 4),
    "Training Time":elapsed_str,
    **{f"hp_{k}": v for k, v in random_search.best_params_.items()}
}])

print("📊 Random Forest Results:")
print(rf_results.T)
print("\n📄 Classification Report:")
print(classification_report(y_test, y_pred_rf))

### Feature Importance

In [ ]:
importances = pd.Series(best_rf.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances.plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("Feature Importance — Random Forest")
plt.xlabel("Importance Score")
plt.grid(True)
plt.tight_layout()
plt.savefig("rf_feature_importance.png", bbox_inches="tight")
plt.show()

### Confusion Matrix & ROC Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Down", "Up"], yticklabels=["Down", "Up"], ax=ax1)
ax1.set_xlabel("Predicted")
ax1.set_ylabel("Actual")
ax1.set_title("Confusion Matrix — Random Forest")

# ROC Curve
y_prob_rf = best_rf.predict_proba(X_test_scaled)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob_rf)
roc_auc = auc(fpr, tpr)
ax2.plot(fpr, tpr, color='blue', lw=2, label=f"ROC (AUC = {roc_auc:.2f})")
ax2.plot([0, 1], [0, 1], color='gray', linestyle='--', label="Random Guess")
ax2.set_xlabel("False Positive Rate")
ax2.set_ylabel("True Positive Rate")
ax2.set_title("ROC Curve — Random Forest")
ax2.legend(loc="lower right")
ax2.grid(True)

plt.tight_layout()
plt.savefig("rf_evaluation.png", bbox_inches="tight")
plt.show()

## 5. Model Training — Logistic Regression

L1-regularized logistic regression as a baseline for comparison.

In [ ]:
log_reg = LogisticRegression(penalty='l1', solver='liblinear', class_weight='balanced')
log_reg.fit(X_train_resampled, y_train_resampled)

y_pred_lr = log_reg.predict(X_test_scaled)
y_prob_lr  = log_reg.predict_proba(X_test_scaled)[:, 1]

lr_results = pd.DataFrame([{
    "Model":     "Logistic Regression",
    "Accuracy":  round(accuracy_score(y_test, y_pred_lr), 4),
    "Precision": round(precision_score(y_test, y_pred_lr, zero_division=0), 4),
    "Recall":    round(recall_score(y_test, y_pred_lr, zero_division=0), 4),
    "F1 Score":  round(f1_score(y_test, y_pred_lr, zero_division=0), 4),
}])

print("📊 Logistic Regression Results:")
print(lr_results.T)
print("\n📄 Classification Report:")
print(classification_report(y_test, y_pred_lr))

# Coefficients
coef_df = pd.DataFrame({
    "Feature":        feature_cols,
    "Coefficient (β)":log_reg.coef_[0]
}).sort_values("Coefficient (β)", ascending=False)
print("\n📈 Logistic Regression Coefficients:")
print(coef_df)

## 6. Results

Side-by-side comparison of both models.

In [ ]:
summary = pd.concat([rf_results[["Model","Accuracy","Precision","Recall","F1 Score"]],
                     lr_results], ignore_index=True)
print("📊 Model Comparison:")
print(summary.to_string(index=False))

---
## 📝 Key Findings

| # | Finding |
|---|---------|
| 1 | **Momentum_5d** and **Volatility_5d** are the top predictors — recent price movement dominates |
| 2 | **SMOTE** oversampling improved recall for the minority class (Down days) |
| 3 | **Random Forest** outperforms Logistic Regression on F1 Score |
| 4 | Macro indicators (CPI, PCE) have lower individual importance but contribute to ensemble performance |
| 5 | Model predicts next-day direction — not suitable for long-horizon forecasting |

> ⚠️ *This project is for educational purposes. Do not use as financial advice.*
